# ACCESS-CSWG Hackathon 2026: getting started on Gadi

This notebook gets you from "I have an NCI account" to "I can run my group's notebook". It sets you up with:
* **pyISSM**, the Python interface to the Ice-sheet and Sea-level System Model (ISSM). The model itself comes
  from the `access-issm` module;
* **ccdtools**, the Python interface to the ACCESS **Cryosphere Community Datapool (CCD)** on Gadi;
* the extra packages the hackathon groups use (geopandas, rasterio, rioxarray, cmocean, …).

**Steps**

1. NCI account and project memberships *(before the hackathon)*
2. Start an ARE JupyterLab session with the right settings
3. In a Terminal: clone the hackathon repository and run the install script
4. Back here: check everything works
5. Go to your group's folder

> Markdown cells (like this one) are instructions. Code cells are run with **Shift + Enter**. Work through them
> in order. Sections 6–8 are reference material to keep open while you work.

---
## 1. NCI account and projects *(do this before the hackathon)*

1. Sign up for an NCI account at **[my.nci.org.au/mancini](https://my.nci.org.au/mancini)**.
2. Request membership of these projects. Approval can take a few days, so do it early:

| Project | Why you need it |
|---|---|
| `au88` | Hackathon project: your working folder `/g/data/au88/<username>` and compute time |
| `av17` | Cryosphere Community Datapool (BedMachine, MEaSUREs, RACMO, …) |
| `xp65` | The `conda/analysis3` Python environment |
| `vk83` | The `access-issm` module (the compiled ISSM model) |

Your **NCI username** (e.g. `abc123`) is what replaces `<username>` everywhere below.

---
## 2. Start an ARE JupyterLab session

1. Go to **[are.nci.org.au](https://are.nci.org.au)** and log in with your NCI username and password.
2. Click **JupyterLab** and fill in the form **exactly** like this. Open *Advanced options* for the module fields:

| Field | Value |
|---|---|
| **Walltime (hours)** | long enough for your work (e.g. `4`; GroupSensitivity may want more) |
| **Queue / Compute size** | `normal`; `small` (2 cpus) is enough for GroupElmer and GroupVisualise, GroupSensitivity should use 4+ cpus |
| **Project** | `au88` |
| **Storage** | `gdata/au88+gdata/av17+gdata/xp65+gdata/vk83+scratch/au88` |
| **Module directories** | `/g/data/vk83/modules /g/data/xp65/public/modules` |
| **Modules** | `access-issm/2025.11.0 conda/analysis3 gdal proj` |

3. Click **Launch**. When the session shows **Running**, click **Open JupyterLab**. Waiting a few minutes in the
   queue is normal.

> **What the settings mean.** *Project* is who pays for the compute. *Storage* lists the disks your session can
> see: missing `gdata/av17` means no datapool, and missing `gdata/au88` means no working folder. *Modules* load the
> ISSM model (`access-issm`), Python (`conda/analysis3`) and the geospatial libraries geopandas and rasterio
> need (`gdal`, `proj`).

---
## 3. In a Terminal: clone the repository and install

Open a Terminal: **File → New Launcher → Terminal**. Type these commands, pressing Enter after each line:

```bash
# 1. Make your hackathon folder and go there
mkdir -p /g/data/au88/$USER
cd /g/data/au88/$USER

# 2. Get the hackathon materials
git clone https://github.com/fmccormack/ACCESS-CSWG-hackathon-2026.git
cd ACCESS-CSWG-hackathon-2026

# 3. Install pyISSM, ccdtools and the extra packages (takes a few minutes)
bash install_hackathon_env.sh
```

`$USER` is filled in automatically with your NCI username. The install script:
* loads `conda/analysis3`, `gdal` and `proj`;
* installs numpy (< 1.26), PyYAML, cmocean, geopandas, rasterio, rioxarray and access-ccdtools into
  `/g/data/au88/$USER/python_packages`;
* clones **pyISSM** and **ccdtools** from GitHub into the same folder and installs them;
* prints the installed versions at the end. Look for **`Install OK`**.

Red text isn't necessarily a problem: look for the word `ERROR`. Run `pwd` if you're not sure where you are.

**Later**, to get updates the organisers push to the hackathon repository:
```bash
cd /g/data/au88/$USER/ACCESS-CSWG-hackathon-2026
git pull
```

---
## 4. Check everything works

**Pick the kernel first:** click the kernel name (top right) and choose the `analysis3` conda environment (no `*`
after the name). Use the same kernel for every notebook in the hackathon.

The packages live in your own folder, so Python has to be told where to look. **The next cell is the snippet to
put at the top of any notebook you write.** The group notebooks already have the equivalent.

In [ ]:
# --- copy this cell to the top of your own notebooks -------------------------
import os, sys
USER = os.environ['USER']
PYBASE = f'/g/data/au88/{USER}/python_packages'
sys.path.insert(0, f'{PYBASE}/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages')
sys.path.append(f'{PYBASE}/pyISSM/src')
sys.path.append(f'{PYBASE}/ccdtools/src')
# -----------------------------------------------------------------------------
print('Python', sys.version.split()[0], '| user', USER)

In [ ]:
# Can we import everything the hackathon needs?
import importlib
packages = ['numpy', 'pandas', 'xarray', 'netCDF4', 'matplotlib', 'scipy', 'dask',
            'geopandas', 'rasterio', 'rioxarray', 'cmocean', 'yaml', 'pyissm', 'ccdtools']
ok = True
for name in packages:
    try:
        mod = importlib.import_module(name)
        print(f'  OK   {name:12s} {getattr(mod, "__version__", "")}')
    except Exception as e:
        ok = False
        print(f'  FAIL {name:12s} {type(e).__name__}: {e}')
print('\nAll packages import' if ok else '\nSome packages are missing: see the troubleshooting table in Section 8')

In [ ]:
# Is the ISSM model available (access-issm module), and can we see the datapool and our folder?
import shutil
checks = {
    'ISSM_DIR set (access-issm loaded)': bool(os.environ.get('ISSM_DIR')),
    'issm.exe on PATH': shutil.which('issm.exe') is not None,
    '/g/data/av17 readable (datapool)': os.access('/g/data/av17', os.R_OK),
    f'/g/data/au88/{USER} writable': os.access(f'/g/data/au88/{USER}', os.W_OK),
}
for k, v in checks.items():
    print(('  OK   ' if v else '  FAIL ') + k)
print('ISSM_DIR =', os.environ.get('ISSM_DIR'))

In [ ]:
# Can ccdtools open the datapool catalog? (needs av17 membership and gdata/av17 in Storage)
import ccdtools
try:
    catalog = ccdtools.catalog.DataCatalog()
    print(f"Catalog OK: {catalog.datasets['dataset'].nunique()} datasets")
    display(catalog.search('bedmachine'))
except Exception as e:
    print('Catalog FAILED:', type(e).__name__, e)

In [ ]:
# Quick pyISSM smoke test: build an empty model and look at its structure
import pyissm
md = pyissm.model.Model()
print('pyISSM', pyissm.__version__)
print('model classes:', ', '.join(md.model_class_names()[:12]), '...')

If every line above says **OK**, you're ready. If not, see **Section 8: Troubleshooting**.

---
## 5. Your group

All the hackathon material lives in the repository you just cloned, `/g/data/au88/$USER/ACCESS-CSWG-hackathon-2026`.

| Group | Folder | Start with | What you'll do |
|---|---|---|---|
| **GroupElmer** | `GroupElmer/` | its `README.md`, then `pytest -q tests/` and `notebooks/elmer_to_pyissm_walkthrough.ipynb` | Convert Elmer/Ice (XIOS UGRID) output into a pyISSM model, resolve the `CHECK:` items, validate against Elmer's scalars |
| **GroupVisualise** | `GroupVisualise/` | `GroupVisualise_ccdtools.ipynb` | Interrogate the Cryosphere Community Datapool with ccdtools and make a publication-quality figure |
| **GroupSensitivity** | `GroupSensitivity/` | `pig_perturbed_ensemble_projections.ipynb` | Build a Pine Island Glacier initial state and run perturbed ensembles to 2300 under low and high emissions |

Each folder also has three slides summarising the task.

> **Work in your own copy.** Before you change anything, make a branch so `git pull` never clashes with your
> edits: `git checkout -b <username>-work`. Write large outputs (models, plots, results) under
> `/g/data/au88/$USER/...`, **not** inside the repository.

---
## 6. Long runs: PBS jobs on Gadi

Most work fits in an ARE session, but long prognostic runs (e.g. GroupSensitivity ensembles to 2300) are better
submitted as PBS jobs. A minimal job script (`run_member.pbs`):

```bash
#!/bin/bash
#PBS -P au88
#PBS -q normal
#PBS -l ncpus=4,mem=16GB,walltime=04:00:00
#PBS -l storage=gdata/au88+gdata/av17+gdata/xp65+gdata/vk83+scratch/au88
#PBS -l wd

module use /g/data/vk83/modules /g/data/xp65/public/modules
module load access-issm/2025.11.0 conda/analysis3 gdal proj
python my_ensemble_script.py     # must start with the path snippet from Section 4
```

Submit with `qsub run_member.pbs`, check with `qstat -u $USER`, and cancel with `qdel <jobid>`.

> **Gadi doesn't support PBS job arrays.** For many members, submit one job per scenario or group (a short
> bash loop around `qsub` is fine), or use `nci-parallel` within one job.

---
## 7. Resources

| Resource | Most useful for | Link |
|---|---|---|
| **pyISSM docs** | API reference for the Python interface | [pyissm.readthedocs.io](https://pyissm.readthedocs.io/latest/) |
| **DeepWiki: pyISSM** | AI-generated overview of how the pyISSM code fits together | [deepwiki.com/ACCESS-NRI/pyISSM](https://deepwiki.com/ACCESS-NRI/pyISSM) |
| **pyISSM tutorials** | Meshing → parameterisation → inversion → forward runs, in order | `tutorials/` in [github.com/ACCESS-NRI/pyISSM](https://github.com/ACCESS-NRI/pyISSM) (already cloned to `python_packages/pyISSM`) |
| **ccdtools docs** | Searching and loading CCD datasets | [ccdtools.readthedocs.io](https://ccdtools.readthedocs.io/latest/) |
| **CCD dataset list** | Which datasets, versions and paths exist | [ACCESS-Hive CCD page](https://access-hive-docs.readthedocs.io/model_evaluation/data/cryosphere_datapool/) |
| **ISSM documentation** | Physics, solvers and model components | [issm.jpl.nasa.gov/documentation](https://issm.jpl.nasa.gov/documentation/) |
| **ISSM source** | Source code, issues | [github.com/ISSMteam/ISSM](https://github.com/ISSMteam/ISSM) |
| **Larour et al. (2012)** | The ISSM paper to cite. Much has changed since, so it's background, not a manual | [doi:10.1029/2011JF002140](https://doi.org/10.1029/2011JF002140) |
| **Gadi user guide** | Storage, modules, PBS queues | [opus.nci.org.au … Gadi User Guide](https://opus.nci.org.au/display/Help/Gadi+User+Guide) |
| **ACCESS-Hive forum** | Questions about pyISSM, ccdtools and Gadi. Search first, then post | [forum.access-hive.org.au](https://forum.access-hive.org.au) |

### Terminal cheat sheet

| Command | What it does |
|---|---|
| `pwd` | Show where you are |
| `ls -lh` | List files with sizes |
| `cd <dir>` / `cd ..` | Change directory / go up one |
| `mkdir -p <dir>` | Make a folder (and parents) |
| `cp -r <src> <dst>` | Copy a file or folder |
| `rm -r <path>` | Delete. **There's no recycle bin** |
| `du -sh <dir>` | Folder size (check before filling `/g/data`) |
| `lquota` | Your project's storage quotas |
| `module list` | Modules currently loaded |

### Git cheat sheet

| Command | What it does |
|---|---|
| `git clone <url>` | Download a repository |
| `git pull` | Get the latest updates |
| `git status` / `git diff` | What has changed |
| `git checkout -b <name>` | New branch for your own work |
| `git add <file>` then `git commit -m "msg"` | Save a snapshot |
| `git push -u origin <name>` | Upload your branch (needs a GitHub account with access) |
| `git log --oneline -10` | Recent history |

**Searching GitHub:** inside a repository's page, the search bar is scoped to that repository. Useful filters:
`is:issue is:closed <error text>` (has someone already fixed this?) and `language:python <function name>`
(examples of a function in use).

---
## 8. Troubleshooting

| Problem | What to check |
|---|---|
| ARE session won't start | Every field in Section 2, especially **Project** (`au88`) and **Storage**. One typo is enough |
| `ModuleNotFoundError` | The kernel is `analysis3`; the path cell in Section 4 ran; `bash install_hackathon_env.sh` printed `Install OK` |
| `geopandas` / `rasterio` import fails | `gdal` and `proj` are in the ARE **Modules** field; re-run the install script |
| `ISSM_DIR` not set / `issm.exe` missing | `access-issm/2025.11.0` in **Modules** and `/g/data/vk83/modules` in **Module directories**; membership of `vk83` |
| Catalog fails / dataset not found | Membership of `av17`; `gdata/av17` in **Storage** |
| `Permission denied` writing files | You're writing outside `/g/data/au88/$USER`. Check the paths in the notebook's settings cell |
| pyISSM `KeyError: 'USER'` | Only outside Gadi: `os.environ.setdefault('USER', 'me')` before importing pyISSM |
| Kernel dies on large datasets | Coarsen or subset before `.compute()` / `.values`; ask for a bigger ARE session |
| Something else | Search the [ACCESS-Hive forum](https://forum.access-hive.org.au), then ask an organiser with a screenshot of the error |